In [1]:

import numpy as np
from sentence_transformers import SentenceTransformer

/Users/harshvardhansingh/Desktop/News_agent/myagenticenv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
encoding = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7523.09it/s]


In [3]:
import json
import feedparser #it is a library that parses RSS feeds and returns a Python object that can be easily manipulated.
with open("feeds.json") as f:
    feeds = json.load(f)["feeds"]


for url in feeds:
    feed = feedparser.parse(url)

    for article in feed.entries:
        print(article.title)
        print(article.link)
        print(article.published)
        print("-" * 40)

Reform receives record £36m donation from crypto billionaire
https://www.bbc.co.uk/news/articles/c3v4zvyde15o?at_medium=RSS&at_campaign=rss
Fri, 11 Sep 2026 23:27:18 GMT
----------------------------------------
Lightning Houthi advance in Yemen may bring dangerous new dimension to Iran war
https://www.bbc.co.uk/news/articles/c3v4zgzr1kxo?at_medium=RSS&at_campaign=rss
Fri, 11 Sep 2026 19:44:58 GMT
----------------------------------------
Masked men, lorries and road blocks: How anti-migrant activists are changing tactics
https://www.bbc.co.uk/news/articles/clyqjgj7x8lo?at_medium=RSS&at_campaign=rss
Fri, 11 Sep 2026 23:06:39 GMT
----------------------------------------
Arrest made over the death of black woman found hanged from tree in Mississippi
https://www.bbc.co.uk/news/articles/cqjkejq5750o?at_medium=RSS&at_campaign=rss
Sat, 12 Sep 2026 02:29:49 GMT
----------------------------------------
MPs vote against fresh attempt to legalise assisted dying
https://www.bbc.co.uk/news/articles/

In [4]:
import sqlite3

conn = sqlite3.connect("news.db")
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS articles (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    title TEXT,
    url TEXT UNIQUE,
    source TEXT,
    published TEXT,
    content TEXT,
    embedded INTEGER DEFAULT 0
)
""")

conn.commit()



In [5]:
cursor.execute("""
SELECT id, url
FROM articles
WHERE content IS NULL
""")
articles= cursor.fetchall() 

from newspaper import Article

for article_id, url in articles:

    art = Article(url)
    art.download()
    art.parse()

    content = art.text

    cursor.execute("""
    UPDATE articles
    SET content = ?
    WHERE id = ?
    """, (content, article_id))

conn.commit()



In [6]:

## chunking the text into smaller pieces for better processing and embedding

from langchain_text_splitters import RecursiveCharacterTextSplitter

def splitter(text, chunk_size=1000, chunk_overlap=200):
    split = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )

    chunks = split.split_text(text)

    ##print(f"Created {len(chunks)} chunks")

    return chunks

In [7]:
from logging import exception
import numpy as np
from sentence_transformers import SentenceTransformer


class EmbeddingManager:

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        print("Initializing embedding model...")
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        try:
            self.model = SentenceTransformer(self.model_name)
        except Exception as e:
            print(f"Error loading model: {e}")

    def Get_embadding(self, texts) -> np.ndarray:
        if not self.model:
            raise ValueError("Model not loaded")
        embedding = self.model.encode(texts, show_progress_bar=True)
        return embedding


    

In [8]:
import time
import chromadb
from newspaper import Article


# =========================================================
# INITIALIZE ONCE
# =========================================================

chroma_client = chromadb.PersistentClient(
    path="./chroma_db"
)

collection = chroma_client.get_or_create_collection(
    name="my_collection_new1"
)

embedding_manager = EmbeddingManager()

EMBED_BATCH_LIMIT = 20  # process at most this many articles per cycle, avoids long first-run blocking


# =========================================================
# HELPER FUNCTIONS
# =========================================================

def get_unembedded_articles(cursor, limit=EMBED_BATCH_LIMIT):
    cursor.execute("""
        SELECT id, content, source, title
        FROM articles
        WHERE embedded = 0
        AND content IS NOT NULL
        LIMIT ?
    """, (limit,))
    return cursor.fetchall()


def mark_embedded(conn, article_id):
    conn.execute(
        "UPDATE articles SET embedded = 1 WHERE id = ?",
        (article_id,)
    )
    conn.commit()


# =========================================================
# MAIN PIPELINE
# =========================================================

def after_10minutes():

    total_start = time.time()

    print("\n" + "=" * 50)
    print("STARTING NEWS INGESTION")
    print("=" * 50)


    # -----------------------------------------------------
    # 1. RSS INGESTION
    # -----------------------------------------------------

    rss_start = time.time()

    print("\n[1] Fetching RSS feeds...")

    new_articles = 0

    for feed_url in feeds:

        try:
            feed = feedparser.parse(feed_url)
            source_name = feed.feed.get("title", feed_url)  # FIX: capture outlet name from the feed

            for article in feed.entries:

                title = article.get("title", "")
                url = article.get("link", "")
                published = article.get("published", "")

                if not url:
                    continue

                cursor.execute("""
                    INSERT OR IGNORE INTO articles
                    (title, url, source, published)
                    VALUES (?, ?, ?, ?)
                """, (
                    title,
                    url,
                    source_name,   # FIX: source now actually stored
                    published
                ))

                if cursor.rowcount > 0:
                    new_articles += 1

        except Exception as e:
            print(f"RSS feed failed: {feed_url}")
            print(f"Error: {e}")

    conn.commit()

    print(f"New articles found: {new_articles}")
    print(
        f"RSS time: {time.time() - rss_start:.2f} seconds"
    )


    # -----------------------------------------------------
    # 2. DOWNLOAD ARTICLE CONTENT
    # -----------------------------------------------------

    download_start = time.time()

    print("\n[2] Downloading article content...")

    cursor.execute("""
        SELECT id, url
        FROM articles
        WHERE content IS NULL
        LIMIT ?
    """, (EMBED_BATCH_LIMIT * 2,))  # cap this stage too, so downloads don't run away on first launch

    articles_to_download = cursor.fetchall()

    print(
        f"Articles to download: "
        f"{len(articles_to_download)}"
    )

    downloaded = 0

    for article_id, url in articles_to_download:

        try:

            art = Article(url)

            art.download()
            art.parse()

            content = art.text

            if not content:
                print(
                    f"No content: article {article_id}"
                )
                continue

            cursor.execute("""
                UPDATE articles
                SET content = ?
                WHERE id = ?
            """, (
                content,
                article_id
            ))

            downloaded += 1

        except Exception as e:

            print(
                f"Failed article {article_id}: {e}"
            )

    conn.commit()

    print(f"Successfully downloaded: {downloaded}")

    print(
        f"Download time: "
        f"{time.time() - download_start:.2f} seconds"
    )


    # -----------------------------------------------------
    # 3. GET UNEMBEDDED ARTICLES
    # -----------------------------------------------------

    embedding_start = time.time()

    print("\n[3] Finding articles to embed...")

    articles = get_unembedded_articles(cursor)

    print(
        f"Articles waiting for embeddings (this cycle, capped at {EMBED_BATCH_LIMIT}): "
        f"{len(articles)}"
    )


    # -----------------------------------------------------
    # 4. CHUNK + EMBED + STORE
    # -----------------------------------------------------

    total_chunks = 0
    embedded_articles = 0

    for article_id, content, source, title in articles:

        try:

            # DIAGNOSTIC: catch abnormally large content (bad scrape) before chunking
            print(
                f"Article {article_id}: content length = {len(content)} chars"
            )
            if len(content) > 20000:
                print(
                    f"  WARNING: unusually long content, likely a bad scrape — skipping"
                )
                mark_embedded(conn, article_id)  # mark done so it doesn't retry forever
                continue

            # Chunk
            chunks = splitter(content)

            if not chunks:
                print(
                    f"No chunks: article {article_id}"
                )
                continue

            print(
                f"Article {article_id}: "
                f"{len(chunks)} chunks"
            )

            total_chunks += len(chunks)


            # Embeddings (progress bar off — adds overhead in a loop)
            embeddings = (
                embedding_manager.Get_embadding(chunks)
            )
            if hasattr(embeddings, "tolist"):
                embeddings = embeddings.tolist()


            # IDs
            ids = [
                f"{article_id}_{i}"
                for i in range(len(chunks))
            ]


            # Metadata
            metadatas = [
                {
                    "article_id": int(article_id),
                    "source": str(source or ""),
                    "title": str(title or "")
                }
                for _ in chunks
            ]


            # Store in ChromaDB
            collection.add(
                embeddings=embeddings,
                documents=chunks,
                metadatas=metadatas,
                ids=ids
            )


            # Only mark after successful Chroma insertion
            mark_embedded(
                conn,
                article_id
            )

            embedded_articles += 1

        except Exception as e:

            print(
                f"Embedding failed for article "
                f"{article_id}: {e}"
            )


    # -----------------------------------------------------
    # 5. SUMMARY
    # -----------------------------------------------------

    embedding_time = time.time() - embedding_start
    total_time = time.time() - total_start

    print("\n" + "=" * 50)
    print("INGESTION COMPLETE")
    print("=" * 50)

    print(
        f"Articles embedded: {embedded_articles}"
    )

    print(
        f"Total chunks: {total_chunks}"
    )

    print(
        f"Embedding time: {embedding_time:.2f} seconds"
    )

    print(
        f"TOTAL TIME: {total_time:.2f} seconds"
    )

    print("=" * 50)

import schedule
import time

# Run immediately
after_10minutes()

# Then every 10 minutes
schedule.every(10).minutes.do(after_10minutes)

while True:
    schedule.run_pending()
    time.sleep(30)

Initializing embedding model...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 14462.63it/s]



STARTING NEWS INGESTION

[1] Fetching RSS feeds...
New articles found: 1
RSS time: 0.62 seconds

[2] Downloading article content...
Articles to download: 1
Successfully downloaded: 1
Download time: 0.51 seconds

[3] Finding articles to embed...
Articles waiting for embeddings (this cycle, capped at 20): 15
Article 161: content length = 2436 chars
Article 161: 4 chunks


Batches: 100%|██████████| 1/1 [00:01<00:00,  1.25s/it]


Article 162: content length = 1988 chars
Article 162: 3 chunks


...

Batches: 100%|██████████| 1/1 [00:00<00:00, 15.71it/s]


Article 178: content length = 1513 chars
Article 178: 2 chunks


Batches: 100%|██████████| 1/1 [00:00<00:00, 26.43it/s]



INGESTION COMPLETE
Articles embedded: 15
Total chunks: 43
Embedding time: 2.76 seconds
TOTAL TIME: 3.89 seconds


KeyboardInterrupt: 

In [9]:
print(collection.count())

167


In [26]:
results = collection.query(
    query_texts=["latest news about india"],
    n_results=5
)

print(results)

{'ids': [['92_5', '161_1', '161_0', '84_0', '168_1']], 'embeddings': None, 'documents': [['"In the absence of a full accountability to the public on behalf of your client, the press fulfils its role and exposes step by step the information to the public," it added.\n\nWhile the UAE has not addressed the conversation specifically, it did not deny it either, issuing a statement emphasising that, since the attack by Hamas, it has worked towards de-escalation, regional stability and preventing violence.\n\n"UAE and Israeli government entities have maintained open and direct lines of communication since the inception of the relationship more than five years ago," the UAE\'s ministry of foreign affairs said in a statement after the publication of Haaretz\'s report. "When necessary, all relevant intelligence has been and continues to be communicated between the relevant entities."\n\nFour of Israel\'s main opposition leaders issued a joint statement demanding an investigation into what happen

In [10]:
data = collection.get(
    include=["documents", "embeddings", "metadatas"]
)

In [11]:
print(data["documents"][0])
print(data["embeddings"][0])
print(data["metadatas"][0])

Labour's regional mayors in England have pledged to cap a new fee on visitors' overnight stays at 5%, after hospitality bosses warned it could put jobs at risk.

Local leaders are set to gain new powers to charge an uncapped levy, dubbed a "tourist tax", as a percentage of the cost of hotels, bed and breakfasts and other types of accommodation.

But the Labour mayors in 10 city regions said they would voluntarily limit the charge if they do apply it, adding that they had listened to concerns over costs.

Reform UK's two regional mayors and the two Conservative ones are likely to oppose any levy given criticism from their national parties of the policy.

In a letter to Chancellor John Healey and Local Government Secretary Angela Rayner, the Labour metro mayors said 5% represented a "reasonable ceiling" on the tax.
[ 1.04149260e-01 -2.85777375e-02  8.48483667e-02  4.80364710e-02
  8.61563459e-02  6.83210744e-03  4.40718681e-02 -5.26708364e-02
 -1.14335567e-01  2.19667684e-02  2.56404150e

# this where i am strat adding tools and llm

In [46]:
import os 
from dotenv import load_dotenv
load_dotenv()

from langchain_tavily import TavilySearch

from langchain_core.tools import tool
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.graph import StateGraph, START, END
from  typing_extensions import TypedDict
from typing import Annotated,Optional
from langgraph.graph.message import add_messages

from langchain.chat_models import init_chat_model



model = init_chat_model(
    "groq:openai/gpt-oss-120b"
)


embedding_model = EmbeddingManager() 

class State(TypedDict):
    messages: Annotated[list, add_messages]
    query: str
    retrieved_docs: list[dict]           # chunks from retrieval
    route_decision: Optional[str]        # "single_search" or "compare_outlets"
    outlet_results: Optional[dict]       # for cross-outlet comparison
    answer: Optional[str]




def retrieve_node(state: State):
    query = state["query"]

    query_embedding = embedding_model.Get_embadding(query)

    if hasattr(query_embedding, "tolist"):
        query_embedding = query_embedding.tolist()

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=5
    )

    print("RAW RESULTS:")
    print(results)

    docs = []

    for text, meta, dist in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0]
    ):
        docs.append({
            "text": text,
            "metadata": meta,
            "distance": dist
        })

    print("RETRIEVED:", len(docs))

    return {"retrieved_docs": docs}

def generate_node(state: State):
    query = state["query"]
    docs = state["retrieved_docs"]

    print("\n" + "=" * 70)
    print("                    🧠 GENERATE NODE")
    print("=" * 70)

    print(f"🔎 Query          : {query}")
    print(f"📚 Documents      : {len(docs)}")

    if not docs:
        print("⚠️  No documents retrieved.")
        print("=" * 70)

        return {
            "answer": "I couldn't find enough relevant information to answer your question.",
            "messages": []
        }

    # -------------------------
    # Retrieved sources
    # -------------------------

    print("\n📑 RETRIEVED SOURCES")
    print("-" * 70)

    for i, doc in enumerate(docs, 1):
        metadata = doc.get("metadata", {})

        title = metadata.get("title", "Unknown title")
        source = metadata.get("source", "Unknown source")
        distance = doc.get("distance")

        print(f"{i}. {title}")
        print(f"   Source   : {source}")
        print(f"   Distance : {distance:.4f}" if distance is not None else
              "   Distance : N/A")

    # -------------------------
    # Build context
    # -------------------------

    context = "\n\n".join(
        f"""
Source: {d['metadata'].get('source', 'Unknown')}
Title: {d['metadata'].get('title', 'Unknown')}
Content:
{d['text']}
"""
        for d in docs
    )

    print("\n📝 CONTEXT PREVIEW")
    print("-" * 70)

    preview = context[:500].replace("\n", " ")
    print(preview + "..." if len(context) > 500 else preview)

    # -------------------------
    # Prompt
    # -------------------------

    prompt = f"""
You are a reliable news assistant.

Answer the user's question using ONLY the provided context.

Rules:
- Do not invent facts.
- Do not use information outside the context.
- If the context does not contain enough information, say so.
- Give a clear and concise answer.
- Mention the source when relevant.

Context:
{context}

Question:
{query}

Answer:
"""

    # -------------------------
    # Generate
    # -------------------------

    print("\n🤖 GENERATING ANSWER...")

    response = model.invoke(prompt)

    answer = response.content

    print("\n💬 ANSWER")
    print("-" * 70)
    print(answer)

    print("=" * 70)

    return {
        "answer": answer,
        "messages": [response]
    }

def fallback_node(state: State):
    query = state["query"]
    
    tavily = TavilySearch(max_results=3)
    results = tavily.invoke({"query": query})
    
    # normalize Tavily's output to the same shape as your Chroma retrieved_docs
    # so generate_node doesn't need to know which source it came from
    docs = []
    for r in results.get("results", []):
        docs.append({
            "text": r.get("content", ""),
            "metadata": {
                "source": r.get("url", "web"),
                "title": r.get("title", "")
            },
            "distance": None  # not applicable for web search results
        })
    
    return {"retrieved_docs": docs}

def is_retrieval_sufficient(state: State) -> str:
    docs = state["retrieved_docs"]

    if not docs:
        return "fallback"

    return "generate"




def tool_calling (query):
    return {"messages" :model.invoke(query)}

grapghbuilder = StateGraph(State) 


grapghbuilder.add_node("retreiver",retrieve_node)
grapghbuilder.add_node("generate_answer",generate_node)
grapghbuilder.add_node("web_search",fallback_node)




grapghbuilder.add_edge(START,"retreiver")
grapghbuilder.add_conditional_edges(
    "retreiver",
    is_retrieval_sufficient,
    {
        "generate": "generate_answer",
        "fallback": "web_search"
    }
)

grapghbuilder.add_edge("web_search", "generate_answer")
grapghbuilder.add_edge("generate_answer", END)

app = grapghbuilder.compile()




Initializing embedding model...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7529.78it/s]


In [48]:
result = app.invoke({"query": "latest news about israel ", "messages": []})
print(result["answer"])

Batches: 100%|██████████| 1/1 [00:00<00:00,  2.98it/s]


RAW RESULTS:
{'ids': [['92_5', '92_1', '92_0', '92_3', '92_2']], 'embeddings': None, 'documents': [['"In the absence of a full accountability to the public on behalf of your client, the press fulfils its role and exposes step by step the information to the public," it added.\n\nWhile the UAE has not addressed the conversation specifically, it did not deny it either, issuing a statement emphasising that, since the attack by Hamas, it has worked towards de-escalation, regional stability and preventing violence.\n\n"UAE and Israeli government entities have maintained open and direct lines of communication since the inception of the relationship more than five years ago," the UAE\'s ministry of foreign affairs said in a statement after the publication of Haaretz\'s report. "When necessary, all relevant intelligence has been and continues to be communicated between the relevant entities."\n\nFour of Israel\'s main opposition leaders issued a joint statement demanding an investigation into w

In [32]:
import sqlite3
conn = sqlite3.connect("news.db")
print(conn.execute("SELECT COUNT(*) FROM articles").fetchone())
print(conn.execute("SELECT COUNT(*) FROM articles WHERE content IS NOT NULL").fetchone())
print(conn.execute("SELECT COUNT(*) FROM articles WHERE embedded = 1").fetchone())

(104,)
(104,)
(104,)


In [ ]:
import schedul
import time 

def get_updated_news():
    
